<a href="https://colab.research.google.com/github/icosahedron31/Facial-Expression-Recognition/blob/main/VisionTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle wandb onnx -Uq


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
! mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [ ]:
! mkdir -p ~/.kaggle && echo KGAT_8cb0c83c63cdaf3612ac74923b3aa519 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download challenges-in-representation-learning-facial-expression-recognition-challenge

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
replace example_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tdola23 (tdola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Imports**

In [ ]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


**Read and split dataset**

In [ ]:
df = pd.read_csv('../content/train.csv')
train_ds, test_ds = train_test_split(df, test_size = 0.3, random_state=42)
test_ds, val_ds = train_test_split(test_ds, test_size = 0.5, random_state=42)
train_ds.head(), val_ds.head()

(       emotion                                             pixels
 14421        0  157 154 157 137 94 99 115 104 120 110 128 98 1...
 9226         4  11 13 17 20 20 21 24 29 28 28 30 33 40 50 57 6...
 994          2  242 250 154 76 80 69 57 53 50 59 55 45 46 49 5...
 28675        0  111 111 112 110 111 111 109 106 99 88 44 68 12...
 4600         3  7 1 4 17 9 11 25 30 33 31 23 38 29 42 56 48 60...,
        emotion                                             pixels
 17147        3  136 136 137 136 138 137 137 135 133 102 52 36 ...
 14040        3  45 26 33 65 47 36 37 35 36 66 88 69 53 38 47 3...
 21306        3  80 78 78 77 76 76 80 115 132 111 93 80 86 92 8...
 22365        3  250 250 250 251 251 251 252 159 58 69 50 44 42...
 22196        1  255 255 255 255 253 255 165 85 62 57 63 72 90 ...)

**Transform**

In [ ]:
from PIL import Image
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05)
    ),
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])

**Creating Dataset**

In [ ]:
class CustomImageDataset(Dataset):
  def __init__(self, df, transform=None) :
    self.df = df
    self.transform = transform
  def __getitem__(self, index) :
    rows = self.df.iloc[index]
    pixels = rows["pixels"]
    label = torch.tensor(rows["emotion"], dtype=torch.long).to(device)
    pixels = np.fromstring(pixels, dtype=np.uint8, sep=" ").reshape(48, 48)
    pixels = self.transform(pixels).to(device)
    return pixels, label
  def __len__(self) :
    return len(self.df)

In [ ]:
train_ds = CustomImageDataset(train_ds, transform)
val_ds = CustomImageDataset(val_ds, val_transform)

**Model**

In [ ]:
import torch
import torch.nn as nn

class ViTSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_extractor = nn.Conv2d(in_channels = 1, out_channels = 512, kernel_size = 3, stride = 3)
        self.layer = nn.TransformerEncoderLayer(
            d_model = 512,
            dropout=0.0,
            nhead = 1,
            dim_feedforward=1024,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.pool = nn
        self.linear = nn.Linear(512, 7)
    def forward(self, x):

        x = self.patch_extractor(x) #B, E, H, W
        x = x.flatten(2) #B, E, H * W
        x = x.transpose(1, 2) #B, H * W, E
       # print(x.shape)
        x = self.transformer_encoder(x)
       # print(x.shape)
        x = x.mean(dim = 1)
        x = self.linear(x)

       # print(x.shape)
        return x

In [ ]:
import torch
import torch.nn as nn

class ViTPositional(nn.Module):
    def __init__(self, n_embd = 256, num_layers = 3, dropout=0.3, n_head = 1):
        super().__init__()
        self.n_embd = n_embd
        self.num_layers = 3,
        self.dropout = dropout
        self.n_head = n_head
        self.patch_extractor = nn.Conv2d(in_channels = 1, out_channels = self.n_embd, kernel_size = 3, stride = 3)
        self.layer = nn.TransformerEncoderLayer(
            d_model = self.n_embd,
            dropout= self.dropout,
            nhead = self.n_head,
            dim_feedforward=512,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)
        self.pool = nn
        self.linear = nn.Linear(self.n_embd, 7)
        self.pos = nn.Parameter(torch.randn(1,  256, self.n_embd) * 0.2)
    def forward(self, x):

        x = self.patch_extractor(x) #B, E, H, W

        x = x.flatten(2) #B, E, H * W
        x = x.transpose(1, 2) #B, H * W, E
        x = x + self.pos;
       # print(x.shape)
        x = self.transformer_encoder(x)
       # print(x.shape)
        x = x.mean(dim = 1)
        x = self.linear(x)

       # print(x.shape)
        return x

**Training Loop**

In [ ]:
import copy
def train_loop(train_ds, val_ds, model_name, learning_rate, batch_size, model, EPOCHS=30,
               weight_decay=None) :

  train_dataloader = DataLoader(train_ds, batch_size, shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size, shuffle=False)
  criterion = nn.CrossEntropyLoss()
  best_model = model
  best_acc_val = 0
  if weight_decay :
    optimizer = Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  else :
    optimizer = Adam(model.parameters(), lr = learning_rate)
  for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0
    model.train()
    for inputs, labels in train_dataloader:
      optimizer.zero_grad()
      outputs = model(inputs).to(device)
      train_loss = criterion(outputs, labels)
      total_loss_train += train_loss.item()
      train_loss.backward()

      train_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
      total_acc_train += train_acc
      optimizer.step()
    model.eval()
    with torch.no_grad():
      for inputs, labels in val_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        val_loss = criterion(outputs, labels)
        total_loss_val += val_loss.item()

        val_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
        total_acc_val += val_acc
    if total_acc_val > best_acc_val :
        best_acc_val = total_acc_val
        best_model = copy.deepcopy(model.state_dict())
        if epoch > 10 :
          torch.save(best_model, f"{model_name}.pt")

    print(f'''Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/train_ds.__len__(), 4)} Train Accuracy {round((total_acc_train)/train_ds.__len__() * 100, 4)}
                Validation Loss: {round(total_loss_val/val_ds.__len__(), 4)} Validation Accuracy: {round((total_acc_val)/val_ds.__len__() * 100, 4)}''')


    wandb.log({
        "epoch": epoch,
        "train_loss": total_loss_train / train_ds.__len__(),
        "train_acc": round((total_acc_train)/train_ds.__len__() * 100, 4),
        "val_loss": round(total_loss_val/val_ds.__len__(), 4),
        "val_acc": round((total_acc_val)/val_ds.__len__() * 100, 4)
    })

  return best_model

**Training**

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = ViTSmall().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="overfitting_small_ds_simple_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleViT",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "ViT"
        }
    }
)
model = train_loop(small_ds, small_ds, "ViT", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


/tmp/ipykernel_18096/2079142288.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Subset

model = ViTSmall().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="simple_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleTransformer",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "cnn_transformer", 0.0001, 32, model, 30)
torch.save(model, "small_vit.pt")
wandb.finish()


In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = ViTPositional().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="overfitting_small_ds_positional_numlayer3",
    config = {
    "project": "ml_hw_04",
    "model": "SimpleViTPositional",
    "input": {
        "subset_size": 100,
        "dataset": "train_ds"
    },
    "training": {
        "epochs": 100,
        "batch_size": 8,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "overfit_small_dataset",
            "group": "ViT"
        }
    }
)
model = train_loop(small_ds, small_ds, "ViT", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


In [ ]:
from torch.utils.data import Subset

model = ViTPositional().to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="positional_numlayer3_data_augment_and_dropout",
    config = {
    "project": "ml_hw_04",
    "model": "ViTPositional",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "adam"
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "ViT", 0.0001, 32, model, 10)
torch.save(model, "positional_vit.pt")
wandb.finish()


/tmp/ipykernel_18096/387789291.py:19: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.layer, num_layers=3)


Epoch 1/10, Train Loss: 0.0563 Train Accuracy 24.3083
                Validation Loss: 0.0569 Validation Accuracy: 25.238
Epoch 2/10, Train Loss: 0.0559 Train Accuracy 25.408
                Validation Loss: 0.0556 Validation Accuracy: 25.8881
Epoch 3/10, Train Loss: 0.0551 Train Accuracy 26.8859
                Validation Loss: 0.0548 Validation Accuracy: 28.4421
Epoch 4/10, Train Loss: 0.0543 Train Accuracy 28.9311
                Validation Loss: 0.054 Validation Accuracy: 30.8335
Epoch 5/10, Train Loss: 0.0536 Train Accuracy 30.6379
                Validation Loss: 0.0528 Validation Accuracy: 32.8535
Epoch 6/10, Train Loss: 0.053 Train Accuracy 32.0462
                Validation Loss: 0.0525 Validation Accuracy: 34.4555
Epoch 7/10, Train Loss: 0.0523 Train Accuracy 33.4395
                Validation Loss: 0.0527 Validation Accuracy: 34.6645
Epoch 8/10, Train Loss: 0.0518 Train Accuracy 34.1561
                Validation Loss: 0.0515 Validation Accuracy: 35.7557
Epoch 9/10, Train Lo

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▃▄▅▆▆▇▇█
train_loss,█▇▆▅▅▄▃▂▂▁
val_acc,▁▁▃▄▅▆▆▆▇█
val_loss,█▇▆▅▄▄▄▃▃▁
epoch,9
train_acc,36.2908
train_loss,0.05062
val_acc,39.3778
val_loss,0.0493


In [ ]:
from torch.utils.data import Subset

model = ViTPositional(n_embd=128, dropout=0.1, num_layers = 3, n_head=4).to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="positional_numlayer3_data_augment_and_dropout",
    config = {
    "project": "ml_hw_04",
    "model": "ViTPositional",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 5e-4,
        "optimizer": "adam",
        "n_head": 4,
        "num_layers": 3,
        "n_embd": 128,
        "dropout": 0.1
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "ViT", 0.0005, 32, model, 10)
torch.save(model, "positional_vit.pt")
wandb.finish()


Epoch 1/10, Train Loss: 0.0565 Train Accuracy 24.1292
                Validation Loss: 0.0562 Validation Accuracy: 25.563
Epoch 2/10, Train Loss: 0.0558 Train Accuracy 25.622
                Validation Loss: 0.0554 Validation Accuracy: 27.7687
Epoch 3/10, Train Loss: 0.0546 Train Accuracy 28.1698
                Validation Loss: 0.0534 Validation Accuracy: 31.3908
Epoch 4/10, Train Loss: 0.0532 Train Accuracy 31.6879
                Validation Loss: 0.0527 Validation Accuracy: 33.1089
Epoch 5/10, Train Loss: 0.052 Train Accuracy 33.5092
                Validation Loss: 0.0515 Validation Accuracy: 34.7342
Epoch 6/10, Train Loss: 0.051 Train Accuracy 35.0169
                Validation Loss: 0.0494 Validation Accuracy: 37.915
Epoch 7/10, Train Loss: 0.0503 Train Accuracy 36.1067
                Validation Loss: 0.0502 Validation Accuracy: 37.056
Epoch 8/10, Train Loss: 0.0496 Train Accuracy 37.3756
                Validation Loss: 0.0504 Validation Accuracy: 37.59
Epoch 9/10, Train Loss: 

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▃▄▅▆▇▇██
train_loss,█▇▆▅▄▃▃▂▂▁
val_acc,▁▂▄▄▅▆▆▆██
val_loss,█▇▆▅▄▂▃▃▁▁
epoch,9
train_acc,39.2765
train_loss,0.04843
val_acc,41.1191
val_loss,0.0477


In [ ]:
from torch.utils.data import Subset

model = ViTPositional(n_embd=128, dropout=0.1, num_layers = 3, n_head=4).to(device)

wandb.init(
    project="ml_hw_04",
    group="ViT",
    name="positional_numlayer3_data_augment_and_dropout",
    config = {
    "project": "ml_hw_04",
    "model": "ViTPositional",

    "training": {
        "epochs": 30,
        "batch_size": 32,
        "lr": 1e-3,
        "optimizer": "adam",
        "n_head": 4,
        "num_layers": 3,
        "n_embd": 128,
        "dropout": 0.1
    },
        "experiment": {
            "goal": "train_transformer",
            "group": "ViT"
        }
    }
)
model = train_loop(train_ds, val_ds, "ViT", 0.001, 32, model, 10)
torch.save(model, "positional_vit.pt")
wandb.finish()


Epoch 1/10, Train Loss: 0.0566 Train Accuracy 23.9401
                Validation Loss: 0.0564 Validation Accuracy: 25.1451
Epoch 2/10, Train Loss: 0.0561 Train Accuracy 24.8557
                Validation Loss: 0.0562 Validation Accuracy: 25.8184
Epoch 3/10, Train Loss: 0.0558 Train Accuracy 25.2986
                Validation Loss: 0.0563 Validation Accuracy: 25.4934
Epoch 4/10, Train Loss: 0.0559 Train Accuracy 25.0995
                Validation Loss: 0.0559 Validation Accuracy: 25.563
Epoch 5/10, Train Loss: 0.0557 Train Accuracy 25.7365
                Validation Loss: 0.0556 Validation Accuracy: 26.6775
Epoch 6/10, Train Loss: 0.0562 Train Accuracy 25.1791
                Validation Loss: 0.0561 Validation Accuracy: 25.3541
Epoch 7/10, Train Loss: 0.0557 Train Accuracy 25.9504
                Validation Loss: 0.0561 Validation Accuracy: 26.7007
Epoch 8/10, Train Loss: 0.0555 Train Accuracy 26.5326
                Validation Loss: 0.0561 Validation Accuracy: 26.1435
Epoch 9/10, Train

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▅▄▆▄▆█▇▇
train_loss,█▅▃▃▃▆▂▁▂▁
val_acc,▃▅▄▅█▄█▆▅▁
val_loss,▄▃▃▂▁▃▃▃▂█
epoch,9
train_acc,26.3037
train_loss,0.05555
val_acc,24.3325
val_loss,0.0576
